# SOP Monitoring Training Service — Agentic Fine-Tuning Walkthrough

This notebook prepares the **Server Fan Installation** sample dataset for the **agentic** fine-tuning flow and shows you how to hand the rest of the work off to an agent (`/sop-ft-orchestrate`).

This notebook is intentionally short: it **downloads + stages the dataset** and stops there. The actual fine-tuning, evaluation, and RCA steps are driven by skills installed into your agent — see [`AGENTIC_README.md`](../AGENTIC_README.md) at the repo root for the skill reference and install instructions.

By the end of this notebook you'll have:

- `assets/data/server_fan_agentic_train/` — 10 videos with per-video annotations
- `assets/data/server_fan_agentic_test/` — 2 held-out videos

…in the exact layout the BP services and the orchestrator expect.

> The dataset names end in `_agentic_train` / `_agentic_test` purely for readability — they keep this walkthrough's artifacts separate from anything you may already have staged from the manual notebook.

## Prerequisites

- **Training BP running**: `docker compose up` is healthy. The orchestrator and individual skills both talk to these services over HTTP.
- **NGC CLI configured**: this notebook downloads the sample dataset using the [NGC CLI](https://org.ngc.nvidia.com/setup/installers/cli). Run `ngc config set` once with credentials that can read `nvidia/tao/sop-server-fan-installation-data:1.0-260213`. If the dataset is already extracted at `sop_data/sop-server-fan-installation-data_v1.0-260213/`, the download step is a no-op.
- **Plugins installed into your agent**: follow the *Installing the plugins* section in [`AGENTIC_README.md`](../AGENTIC_README.md).

## 0. Paths

Everything in this notebook is expressed relative to the repo root, so it works regardless of where Jupyter was launched.

In [ ]:
import os

NOTEBOOK_DIR = os.path.abspath(os.getcwd())
REPO_ROOT    = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))

# --- Source: NGC sample dataset (downloaded by Section 1 below) ---
SOP_DATA_DIR     = os.path.join(REPO_ROOT, "sop_data")
DOWNLOAD_DIR     = os.path.join(SOP_DATA_DIR, "sop-server-fan-installation-data_v1.0-260213")
SAMPLE_DATA_ROOT = os.path.join(DOWNLOAD_DIR, "server_fan")
ARCHIVE_PATH     = os.path.join(DOWNLOAD_DIR, "sop-sample-training-data.tar.gz")

SRC_TRAIN_DIR = os.path.join(SAMPLE_DATA_ROOT, "train")  # has actions.json + per-video chunk folders
SRC_TEST_DIR  = os.path.join(SAMPLE_DATA_ROOT, "test")   # has per-video chunk folders, no actions.json
SRC_RAW_DIR   = os.path.join(SAMPLE_DATA_ROOT, "raw")    # Install_*.MP4 (full-length videos)

# Per-video annotation JSONs live next to this notebook (mirror of what the Annotation UI would emit)
SAMPLE_ANNOTATIONS_DIR = os.path.join(NOTEBOOK_DIR, "sample_data_annotation")
SRC_TRAIN_ANN_DIR = os.path.join(SAMPLE_ANNOTATIONS_DIR, "train")
SRC_TEST_ANN_DIR  = os.path.join(SAMPLE_ANNOTATIONS_DIR, "test")

# --- Destination: where the BP services read datasets from ---
ASSETS_DATA_DIR = os.path.join(REPO_ROOT, "assets", "data")
TRAIN_DATASET_ID = "server_fan_agentic_train"
TEST_DATASET_ID  = "server_fan_agentic_test"
DST_TRAIN_DIR = os.path.join(ASSETS_DATA_DIR, TRAIN_DATASET_ID)
DST_TEST_DIR  = os.path.join(ASSETS_DATA_DIR, TEST_DATASET_ID)

TRAIN_VIDEO_IDS = [f"Install_{i}" for i in [1, 3, 4, 5, 6, 7, 8, 9, 10, 11]]
TEST_VIDEO_IDS  = ["Install_12", "Install_13"]

print(f"REPO_ROOT        -> {REPO_ROOT}")
print(f"SOP_DATA_DIR     -> {SOP_DATA_DIR}")
print(f"SAMPLE_DATA_ROOT -> {SAMPLE_DATA_ROOT}")
print(f"ASSETS_DATA_DIR  -> {ASSETS_DATA_DIR}")

## 1. Download and extract the sample dataset

Downloads `nvidia/tao/sop-server-fan-installation-data:1.0-260213` from NGC into `sop_data/` and extracts the `server_fan/` directory. After extraction:

```
sop_data/sop-server-fan-installation-data_v1.0-260213/server_fan/
├── raw/         # Install_1.MP4 ... Install_13.MP4  (full-length source videos)
├── train/       # actions.json + per-video chunk folders for training
├── test/        # held-out videos + labels.txt
└── augmented/   # pre-generated QA data (not used by the agentic flow)
```

The cell is idempotent: if `server_fan/` already exists it's a no-op; if only the tarball is present, it just re-extracts. The download is several GB, so the first run takes a while.

In [ ]:
import shutil
import subprocess
import tarfile

NGC_RESOURCE = "nvidia/tao/sop-server-fan-installation-data:1.0-260213"


def _ngc_download():
    os.makedirs(SOP_DATA_DIR, exist_ok=True)
    print(f"Downloading {NGC_RESOURCE} into {SOP_DATA_DIR} ...")
    # The NGC CLI creates a directory named '<resource>_v<version>' in cwd.
    result = subprocess.run(
        ["ngc", "registry", "resource", "download-version", NGC_RESOURCE],
        cwd=SOP_DATA_DIR,
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(
            "NGC download failed. Make sure the NGC CLI is installed and `ngc config set` "
            "has been run with credentials that can access the resource."
        )


def _extract_archive():
    if not os.path.isfile(ARCHIVE_PATH):
        raise FileNotFoundError(f"Expected archive not found: {ARCHIVE_PATH}")
    print(f"Extracting {ARCHIVE_PATH} ...")
    with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
        tar.extractall(path=DOWNLOAD_DIR, filter="data")


if os.path.isdir(SAMPLE_DATA_ROOT):
    print(f"Dataset already extracted at {SAMPLE_DATA_ROOT} - skipping download/extract.")
elif os.path.isfile(ARCHIVE_PATH):
    print(f"Archive already present at {ARCHIVE_PATH} - extracting only.")
    _extract_archive()
else:
    if shutil.which("ngc") is None:
        raise RuntimeError(
            "The `ngc` CLI was not found on PATH. Install it from "
            "https://org.ngc.nvidia.com/setup/installers/cli and run `ngc config set`."
        )
    _ngc_download()
    _extract_archive()

# Verify the layout we'll consume in Section 2.
for path, label in [
    (SAMPLE_DATA_ROOT,        "server_fan/"),
    (SRC_TRAIN_DIR,           "server_fan/train/"),
    (SRC_TEST_DIR,            "server_fan/test/"),
    (SRC_RAW_DIR,             "server_fan/raw/"),
    (SAMPLE_ANNOTATIONS_DIR,  "tutorials/sample_data_annotation/"),
]:
    marker = "OK " if os.path.exists(path) else "!! MISSING -- "
    print(f"{marker}{label:35s} -> {path}")

## 2. Stage the dataset under `assets/data/`

The agentic flow reads datasets from `assets/data/<dataset_id>/`. The expected layout for each dataset is:

```
assets/data/<dataset_id>/
├── actions.json                    # action ID -> description
├── <Install_N>.mp4                 # full-length video (lower-case .mp4)
├── <Install_N>/
│   ├── *_annotation.json           # per-video timestamp annotation
│   └── *.mp4                       # chunk videos
└── ...
```

The source dataset (`sop_data/.../server_fan/`) is *close* to this layout but needs four minor fixes:

1. `train/actions.json` is missing the new `actions_can_be_skipped` field
2. `test/` doesn't have its own `actions.json` (we mirror it from train)
3. The full-length videos live in `raw/Install_*.MP4` (upper-case) and need to be present in each split as lower-case `<vid>.mp4`
4. The per-video `*_annotation.json` files live in `tutorials/sample_data_annotation/` and need to land inside each video's subfolder

The cells below do these in order. Every step is idempotent — re-running is a no-op once the staged data is in place.

### 2.1 Mirror the train and test splits into `assets/data/`

Copies `sop_data/.../server_fan/{train,test}/` into `assets/data/server_fan_agentic_{train,test}/`. Chunk videos and any configs already shipped in the source dataset come along for the ride.

In [ ]:
os.makedirs(ASSETS_DATA_DIR, exist_ok=True)

def _copy_if_missing(src, dst):
    if os.path.exists(dst):
        return False
    if os.path.isdir(src):
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    return True

def mirror_split(src_dir, dst_dir):
    os.makedirs(dst_dir, exist_ok=True)
    new_entries = []
    for entry in sorted(os.listdir(src_dir)):
        if _copy_if_missing(os.path.join(src_dir, entry), os.path.join(dst_dir, entry)):
            new_entries.append(entry)
    return new_entries

print(f"Mirror train -> {DST_TRAIN_DIR}")
added = mirror_split(SRC_TRAIN_DIR, DST_TRAIN_DIR)
print(f"  added {len(added)} entries" + (f": {added}" if added else " (already staged)"))

print(f"\nMirror test  -> {DST_TEST_DIR}")
added = mirror_split(SRC_TEST_DIR, DST_TEST_DIR)
print(f"  added {len(added)} entries" + (f": {added}" if added else " (already staged)"))

### 2.2 Patch `actions.json` (add `actions_can_be_skipped`)

The current pipeline expects an `actions_can_be_skipped` field listing "padding" actions that should be ignored during SOP evaluation. The sample dataset ships without it, so we add the single non-SOP padding step here. A one-time backup of the original is written next to it as `actions.json.bak`.

In [ ]:
import json

ACTIONS_FILENAME = "actions.json"
TRAIN_ACTIONS_JSON   = os.path.join(DST_TRAIN_DIR, ACTIONS_FILENAME)
TRAIN_ACTIONS_BACKUP = TRAIN_ACTIONS_JSON + ".bak"
PADDING_ACTION       = "(10) doing action not belong to the defined SOP."

with open(TRAIN_ACTIONS_JSON) as f:
    actions_data = json.load(f)

if "actions_can_be_skipped" in actions_data:
    print("actions.json already patched -- skipping.")
    print(f"  actions_can_be_skipped = {actions_data['actions_can_be_skipped']}")
else:
    if not os.path.exists(TRAIN_ACTIONS_BACKUP):
        shutil.copy2(TRAIN_ACTIONS_JSON, TRAIN_ACTIONS_BACKUP)
        print(f"Saved backup: {TRAIN_ACTIONS_BACKUP}")
    if PADDING_ACTION not in actions_data["actions"]:
        raise RuntimeError(f"Expected padding action not found in {TRAIN_ACTIONS_JSON}.")
    actions_data["actions_can_be_skipped"] = [PADDING_ACTION]
    with open(TRAIN_ACTIONS_JSON, "w") as f:
        json.dump(actions_data, f, indent=4)
    print(f"Patched {TRAIN_ACTIONS_JSON}: actions_can_be_skipped = {actions_data['actions_can_be_skipped']}")

### 2.3 Copy `actions.json` to the test split

The test split needs the same action definitions so the orchestrator can validate predictions against them.

In [ ]:
TEST_ACTIONS_JSON = os.path.join(DST_TEST_DIR, ACTIONS_FILENAME)
shutil.copy2(TRAIN_ACTIONS_JSON, TEST_ACTIONS_JSON)  # always overwrite so the patched version wins
print(f"Copied actions.json -> {TEST_ACTIONS_JSON}")

### 2.4 Place the full-length videos

DDM training and the orchestrator both look for the full-length raw video at `<dataset>/<Install_N>.mp4` (lower-case `.mp4`). The NGC archive ships them under `raw/Install_*.MP4`; we copy the right subset into each split.

In [ ]:
ALREADY_PRESENT = "(already present)"

def copy_raw_videos(video_ids, dst_dir):
    copied = []
    for vid in video_ids:
        src = os.path.join(SRC_RAW_DIR, f"{vid}.MP4")
        dst = os.path.join(dst_dir, f"{vid}.mp4")
        if not os.path.isfile(src):
            print(f"  WARN: missing raw video: {src}")
            continue
        if _copy_if_missing(src, dst):
            copied.append(vid)
    return copied

print(f"Copy raw videos -> {DST_TRAIN_DIR}")
added = copy_raw_videos(TRAIN_VIDEO_IDS, DST_TRAIN_DIR)
print(f"  copied {len(added)} " + (f"({added})" if added else ALREADY_PRESENT))

print(f"\nCopy raw videos -> {DST_TEST_DIR}")
added = copy_raw_videos(TEST_VIDEO_IDS, DST_TEST_DIR)
print(f"  copied {len(added)} " + (f"({added})" if added else ALREADY_PRESENT))

### 2.5 Drop per-video annotation JSONs into each video's folder

The annotation backend (and any skill that imports a dataset) reads one `*_annotation.json` per video, expected to live *inside* that video's chunk folder (`<dataset>/<Install_N>/<Install_N>_annotation.json`). The source files come from `tutorials/sample_data_annotation/`.

In [ ]:
def place_annotations(video_ids, src_ann_dir, dst_dataset_dir):
    placed = []
    for vid in video_ids:
        src_ann = os.path.join(src_ann_dir, f"{vid}_annotation.json")
        dst_video_dir = os.path.join(dst_dataset_dir, vid)
        if not os.path.isfile(src_ann):
            print(f"  WARN: annotation missing: {src_ann}")
            continue
        os.makedirs(dst_video_dir, exist_ok=True)
        if _copy_if_missing(src_ann, os.path.join(dst_video_dir, f"{vid}_annotation.json")):
            placed.append(vid)
    return placed

print(f"Annotations -> {DST_TRAIN_DIR}")
added = place_annotations(TRAIN_VIDEO_IDS, SRC_TRAIN_ANN_DIR, DST_TRAIN_DIR)
print(f"  placed {len(added)} " + (f"({added})" if added else ALREADY_PRESENT))

print(f"\nAnnotations -> {DST_TEST_DIR}")
added = place_annotations(TEST_VIDEO_IDS, SRC_TEST_ANN_DIR, DST_TEST_DIR)
print(f"  placed {len(added)} " + (f"({added})" if added else ALREADY_PRESENT))

### 2.6 Sanity check the final layout

A pass/fail summary of what the orchestrator and skills will see when they read these dataset directories.

In [ ]:
def check_dataset(dataset_dir, video_ids):
    print(f"\n=== {dataset_dir} ===")
    rows = [(ACTIONS_FILENAME, os.path.isfile(os.path.join(dataset_dir, ACTIONS_FILENAME)))]
    for vid in video_ids:
        mp4    = os.path.isfile(os.path.join(dataset_dir, f"{vid}.mp4"))
        subdir = os.path.isdir(os.path.join(dataset_dir, vid))
        ann    = os.path.isfile(os.path.join(dataset_dir, vid, f"{vid}_annotation.json"))
        rows.append((f"{vid}.mp4", mp4))
        rows.append((f"{vid}/", subdir))
        rows.append((f"{vid}/{vid}_annotation.json", ann))
    for name, ok in rows:
        print(f"  {'OK ' if ok else 'MISS '}{name}")

check_dataset(DST_TRAIN_DIR, TRAIN_VIDEO_IDS)
check_dataset(DST_TEST_DIR,  TEST_VIDEO_IDS)

## 3. Drive the agentic fine-tuning flow

From here on, the Python kernel sits idle — the actual fine-tuning is driven by your agent. Open Claude Code (or your framework of choice) from the repo root and pick one of the two paths below.

If you haven't installed the plugins yet, do that first — see *Installing the plugins* in [`AGENTIC_README.md`](../AGENTIC_README.md).

### 3.1 Option A — End-to-end orchestrated run (recommended)

`/sop-ft-orchestrate` handles **dataset import, augmentation, DDM training, VLM training, evaluation, RCA, and config-fix retries** in one closed loop. **You do not need to import the dataset manually first** — the orchestrator detects that `server_fan_agentic_train` is not registered and runs the import as its first step.

Open your agent in this repo and paste:

```
I want to fine-tune SOP monitoring on assets/data/server_fan_agentic_train and evaluate on assets/data/server_fan_agentic_test
targeting seq_accuracy >= 0.90 and max iterations set to 10.
Report status every 10 minutes.
```

The orchestrator writes its outputs to a timestamped directory under the BP's results root (see *What the orchestrator produces* in [`AGENTIC_README.md`](../AGENTIC_README.md)). If the session is interrupted, restart the same command and it'll resume from `run_state.yaml`.

### 3.2 Option B — Drive a single skill (for experimentation)

When you only want to exercise one stage (e.g. regenerate the augmented dataset with different parameters), invoke the individual skill directly. **In this mode the dataset must already be registered in the BP database**, so we run `import_dataset.sh` once before opening the agent.

Run the cell below to register `server_fan_agentic_train` with the annotation backend (no-op if it's already imported and `--force` ensures the metadata is refreshed).

In [ ]:
IMPORT_HELPER = os.path.join(NOTEBOOK_DIR, "helper_scripts", "import_dataset.sh")

def run_import(dataset_id):
    cmd = ["bash", IMPORT_HELPER, dataset_id, "--force"]
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(cmd, check=False)
    if result.returncode != 0:
        raise RuntimeError(
            f"import_dataset.sh failed for '{dataset_id}'. "
            "Is the annotation-backend container running (`docker compose up -d`)?"
        )

run_import(TRAIN_DATASET_ID)
run_import(TEST_DATASET_ID)

Now open your agent and try one of these single-skill prompts (each maps to one BP microservice — see the table in [`AGENTIC_README.md`](../AGENTIC_README.md#skills-in-this-repo)):

**Regenerate the augmented QA dataset with a different config:**

```
/sop-data-augmentation regenerate the augmented dataset for dataset_id=server_fan_agentic_train
with BCQ negative_ratio=1.
```

**Train DDM-Net only:**

```
/sop-ddm-finetuning launch a DDM-Net run on dataset_id=server_fan_agentic_train and val_dataset_id=server_fan_agentic_test
with bilinear interpolation.
```

**Fine-tune the VLM on an existing augmented dataset:**

```
/sop-cr-finetuning train Cosmos-Reason2-2B on qa_augmented_dataset_id=server_fan_agentic_train_augmented_0
with optm_lr=[5e-6, 5e-6, 5e-6].
```

**Run an evaluation against a trained checkpoint:**

```
/sop-e2e-inference evaluate <vlm_checkpoint_path> + <ddm_checkpoint_path>
on the test data server_fan_agentic_test.
```

## 4. End of walkthrough

What you produced:

| Artifact | Location |
|---|---|
| Sample dataset (downloaded) | `sop_data/sop-server-fan-installation-data_v1.0-260213/server_fan/` |
| Train dataset | `assets/data/server_fan_agentic_train/` |
| Test dataset  | `assets/data/server_fan_agentic_test/` |
| Patched `actions.json` (and `.bak` of the original) | `assets/data/server_fan_agentic_train/actions.json{,.bak}` |

Where to look next:

- [`AGENTIC_README.md`](../AGENTIC_README.md) — skill reference, plugin install steps, prompt patterns, success criteria, troubleshooting
- [`sop-agentic-ft/`](../sop-agentic-ft/) — the plugin marketplace; every plugin ships a full `SKILL.md` and any references/scripts it needs
- [`sop_monitoring_training_sample_data.ipynb`](./sop_monitoring_training_sample_data.ipynb) — the manual notebook walkthrough (no agent) for comparison

Orchestrator output (after a run completes) lands under the BP results root in a timestamped `run_<YYYYMMDD_HHMMSS>/` directory — see *What the orchestrator produces* in `AGENTIC_README.md` for the full layout.